In [1]:
import numpy as np
import pandas as pd
from openpyxl import load_workbook
 
 
DATA_COLS = [
    "total_employed", "pct_women", "pct_white",
    "pct_black", "pct_asian", "pct_hispanic",
]
 
 
def _to_numeric(val):
    """Convert a cell value to float, treating dashes and blanks as NaN."""
    if val is None:
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip()
    if s in ("–", "-", "—", ""):
        return np.nan
    return float(s)
 
 
def parse_bls_occupation_file(filepath: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse a BLS CPS Table 11 xlsx file into leaf data and hierarchy tables.
 
    Parameters
    ----------
    filepath : str
        Path to the .xlsx file (e.g. cpsaat11_2024.xlsx).
 
    Returns
    -------
    leaves : pd.DataFrame
        Columns: occupation, total_employed, pct_women, pct_white,
                 pct_black, pct_asian, pct_hispanic, parent_id
    hierarchy : pd.DataFrame
        Columns: id, occupation, level, parent_id, total_employed,
                 pct_women, pct_white, pct_black, pct_asian, pct_hispanic
    """
    wb = load_workbook(filepath)
    ws = wb.active
 
    # --- Step 1: Extract all occupation rows with indent + data ---
    raw_rows = []
    for row_idx in range(1, ws.max_row + 1):
        cell = ws.cell(row=row_idx, column=1)
        val = cell.value
        if not val or not isinstance(val, str) or not val.strip():
            continue
        name = val.strip()
        # Skip header/footer rows
        if name.startswith(("HOUSEHOLD", "[Numbers", "Occupation", "NOTE:", "See")):
            continue
        if name == "Total, 16 years and over":
            continue  # overall total, not part of hierarchy
 
        indent = int(cell.alignment.indent) if cell.alignment else 0
        data = [_to_numeric(ws.cell(row=row_idx, column=c).value) for c in range(2, 8)]
        raw_rows.append({
            "row_idx": row_idx,
            "indent": indent,
            "occupation": name,
            "total_employed": data[0],
            "pct_women": data[1],
            "pct_white": data[2],
            "pct_black": data[3],
            "pct_asian": data[4],
            "pct_hispanic": data[5],
        })
 
    # --- Step 2: Classify each row as parent or leaf ---
    # A parent is any row whose next row has a strictly higher indent.
    for i, row in enumerate(raw_rows):
        if i < len(raw_rows) - 1 and raw_rows[i + 1]["indent"] > row["indent"]:
            row["is_parent"] = True
        else:
            row["is_parent"] = False
 
    # --- Step 3: Assign IDs to parents and build hierarchy ---
    # Walk through rows maintaining a stack of ancestors (one per indent level).
    # Stack maps indent_level -> parent dict.
    parent_id_counter = 0
    ancestor_stack = {}  # indent_level -> parent row dict
    hierarchy_rows = []
 
    for row in raw_rows:
        indent = row["indent"]
 
        if row["is_parent"]:
            parent_id_counter += 1
            row["id"] = f"L{indent}_{parent_id_counter:04d}"
 
            # This parent's own parent is the most recent ancestor
            # at a strictly lower indent level.
            my_parent_id = None
            for lvl in sorted(ancestor_stack.keys(), reverse=True):
                if lvl < indent:
                    my_parent_id = ancestor_stack[lvl]["id"]
                    break
            row["parent_id"] = my_parent_id
 
            # Clear any ancestor entries at this level or deeper
            ancestor_stack = {k: v for k, v in ancestor_stack.items() if k < indent}
            ancestor_stack[indent] = row
 
            hierarchy_rows.append({
                "id": row["id"],
                "occupation": row["occupation"],
                "level": indent,
                "parent_id": my_parent_id,
                "total_employed": row["total_employed"],
                "pct_women": row["pct_women"],
                "pct_white": row["pct_white"],
                "pct_black": row["pct_black"],
                "pct_asian": row["pct_asian"],
                "pct_hispanic": row["pct_hispanic"],
            })
        else:
            # Leaf: find its parent from the ancestor stack
            leaf_parent_id = None
            for lvl in sorted(ancestor_stack.keys(), reverse=True):
                if lvl < indent:
                    leaf_parent_id = ancestor_stack[lvl]["id"]
                    break
            row["parent_id"] = leaf_parent_id
 
    # --- Step 4: Build DataFrames ---
    leaf_rows = [r for r in raw_rows if not r["is_parent"]]
 
    leaves = pd.DataFrame(leaf_rows)[[
        "occupation", "total_employed", "pct_women", "pct_white",
        "pct_black", "pct_asian", "pct_hispanic", "parent_id"
    ]]
 
    hierarchy = pd.DataFrame(hierarchy_rows)[[
        "id", "occupation", "level", "parent_id", "total_employed",
        "pct_women", "pct_white", "pct_black", "pct_asian", "pct_hispanic"
    ]]
 
    return leaves, hierarchy
 
 
 
def get_full_ancestry(leaves: pd.DataFrame, hierarchy: pd.DataFrame) -> pd.DataFrame:
    """
    Convenience: join leaves with all ancestor levels to produce a wide table.
 
    Returns a DataFrame with columns:
        occupation, [data cols], parent_occupation, grandparent_occupation, ...
    up to the number of hierarchy levels present.
    """
    # Build a lookup from hierarchy id -> row
    h = hierarchy.set_index("id")
    max_depth = int(hierarchy["level"].max()) + 1
 
    ancestor_cols = {i: [] for i in range(max_depth)}
 
    for _, leaf in leaves.iterrows():
        pid = leaf["parent_id"]
        visited = {}
        while pid and pid in h.index:
            parent = h.loc[pid]
            visited[int(parent["level"])] = parent["occupation"]
            pid = parent["parent_id"]
        for lvl in range(max_depth):
            ancestor_cols[lvl].append(visited.get(lvl, None))
 
    result = leaves.copy()
    level_names = {0: "major_group", 1: "mid_group", 2: "minor_group"}
    for lvl in range(max_depth):
        col_name = level_names.get(lvl, f"level_{lvl}")
        result[col_name] = ancestor_cols[lvl]
 
    return result
 

In [11]:
for year in range(2015, 2025):
    fp = f"../data/cpsaat11_{year}.xlsx"
    leaves, hierarchy = parse_bls_occupation_file(fp)
    leaves.to_csv(f"../data/bls_occupation_data_{year}.csv", index=False)
    hierarchy.to_csv(f"../data/bls_industry_data_{year}.csv", index=False)
    
    full = get_full_ancestry(leaves, hierarchy)
    full.to_csv(f"../data/bls_combined_data_{year}.csv", index=False)

    all_occupations = sorted(set(leaves["occupation"].tolist()))
    
    with open(f"../data/occupations_{year}.txt", "w") as f:
        f.write("\n".join(all_occupations))



### do we need to worry about providing unique job categories for each year?
Compare the occupations files

In [9]:
import hashlib
from pathlib import Path

files = [f"../data/occupations_{year}.txt" for year in range(2015, 2025)]

hashes = {}
for f in files:
    h = hashlib.md5(Path(f).read_bytes()).hexdigest()
    hashes[f] = h

# Group files by hash
groups = {}
for f, h in hashes.items():
    groups.setdefault(h, []).append(f)

if len(groups) == 1:
    print("All files are identical")
else:
    for h, members in groups.items():
        print(f"Group {h[:8]}: {members}")

All files are identical
